In [1]:
# Warning control
import warnings
from crewai import Agent, Task, Crew

warnings.filterwarnings('ignore')

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")


In [3]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
  model="openai/gpt-oss-120b",
  api_key=api_key, 
  temperature=1,
  top_p=1,
  max_tokens=4096,
)

response = llm.invoke([{"role":"user","content":""}])
if response.additional_kwargs and "reasoning_content" in response.additional_kwargs:
  print(response.additional_kwargs["reasoning_content"])
print(response.content)

We need to respond as ChatGPT. The user sent an empty message? There's no content. Possibly the user just wants to start a conversation. We need to respond politely, ask how can we help. Also maybe mention capabilities. Let's respond friendly.
Hello! 👋  

How can I help you today? If you have a question, need assistance with something, or just want to chat, feel free to let me know!


In [ ]:
from typing import Optional, List
from pydantic import BaseModel, Field


class BibliographyEntry(BaseModel):
    """Json schema for a single bibliography entry."""
    
    entry_type: str = Field(description="The type of publication (e.g., article, inproceedings, book, dataset).")
    citation_key: str = Field(description="A unique citation key, usually formatted as 'PrimaryAuthorYear' (e.g., Smith2024).")
    title: str = Field(description="The exact title of the academic paper.")
    authors: List[str] = Field(description="List of all authors in the format 'Lastname, Firstname'.")
    journal_or_booktitle: Optional[str] = Field(default=None, description="The journal name, conference proceedings, or book title where it was published.")
    year: int = Field(description="The year of publication.")
    volume: Optional[str] = Field(default=None, description="The volume number.")
    number: Optional[str] = Field(default=None, description="The issue number.")
    pages: Optional[str] = Field(default=None, description="The page range (e.g., 100-115).")
    publisher: Optional[str] = Field(default=None, description="The publisher or organization.")
    doi: Optional[str] = Field(default=None, description="The DOI (Digital Object Identifier) if available.")

In [ ]:
from crewai_tools import (
  FileReadTool,
  DirectoryReadTool
)

directory_tool = DirectoryReadTool(directory="./data")
file_read_tool = FileReadTool()


agent_archivist = Agent(
    role="Team Archivist",
    goal="Locate the requested document in the folder and extract its raw content.",
    backstory="""You are an extremely meticulous and agile archivist. Your job is to scan 
    the document directory, find the exact file requested, and read its entire content 
    to make it available to the team.""",
    tools=[directory_tool, file_read_tool],
    max_iterations=2,
    llm=llm,
    verbose=True
)

# Extractor: Specialist in parsing papers and structuring bibliography metadata
agent_extractor = Agent(
    role="Academic Metadata Extractor",
    goal="Extract all relevant bibliographic entities from the paper to generate a structured bibliography entry.",
    backstory="""You are a seasoned academic librarian and research assistant. You have an 
    impeccable eye for identifying scholarly metadata within scientific papers—such as author names, 
    publication years, journal titles, DOIs, and publishers. Your mission is to parse the raw text 
    of a paper, extract these key entities, and organize them perfectly to create flawless 
    bibliographic entries.""",
    max_iterations=2,
    llm=llm,
    verbose=True
)

